Tạo dữ liệu

In [ ]:
import sqlite3
import pandas as pd

# Tạo database in-memory
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Tạo bảng student
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

# Insert dữ liệu
students = [
(1,"Nguyen Minh Hoang","May Tinh",12,6.7),
(2,"Tran Thi Lan","Kinh Te",34,9.2),
(3,"Pham Van Nam","Toan Tin",None,7.9),
(4,"Le Thanh Huyen","Toan Tin",20,7.2),
(5,"Vu Quoc Anh","May Tinh",24,8.0),
(6,"Dang Thuy Linh","May Tinh",24,5.5),
(7,"Bui Tien Dung","Kinh Te",34,9.2),
(8,"Ho Ngoc Mai","Toan Tin",20,8.8),
(9,"Duong Huu Phuc","Kinh Te",None,7.2),
(10,"Cao Thi Hanh","May Tinh",None,7.0)
]

cursor.executemany("INSERT INTO student VALUES (?,?,?,?,?)", students)

# Tạo bảng course
cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

courses = [
(12,"Giai tich"),
(34,"Thong ke"),
(26,"Tin hoc")
]

cursor.executemany("INSERT INTO course VALUES (?,?)", courses)

conn.commit()

1. Join các bảng

a. Tích descartes (cross join)

In [ ]:
df = pd.read_sql("""
SELECT * 
FROM student CROSS JOIN course
""", conn)
print(df)

    student_id               name     class  course_id  score  id course_name
0            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12   Giai tich
1            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  34    Thong ke
2            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  26     Tin hoc
3            2       Tran Thi Lan   Kinh Te       34.0    9.2  12   Giai tich
4            2       Tran Thi Lan   Kinh Te       34.0    9.2  34    Thong ke
5            2       Tran Thi Lan   Kinh Te       34.0    9.2  26     Tin hoc
6            3       Pham Van Nam  Toan Tin        NaN    7.9  12   Giai tich
7            3       Pham Van Nam  Toan Tin        NaN    7.9  34    Thong ke
8            3       Pham Van Nam  Toan Tin        NaN    7.9  26     Tin hoc
9            4     Le Thanh Huyen  Toan Tin       20.0    7.2  12   Giai tich
10           4     Le Thanh Huyen  Toan Tin       20.0    7.2  34    Thong ke
11           4     Le Thanh Huyen  Toan Tin       20.0    7.2  2

Kết quả trên là tích Descartes giữa bảng student và course, trong đó mỗi sinh viên được kết hợp với tất cả các môn học. Điều này làm số dòng tăng lên bằng tích số dòng của hai bảng (10 × 3 = 30).

b. inner join

In [ ]:
df = pd.read_sql("""
SELECT *
FROM student s
INNER JOIN course c
ON s.course_id = c.id
""", conn)
print(df)

   student_id               name     class  course_id  score  id course_name
0           1  Nguyen Minh Hoang  May Tinh         12    6.7  12   Giai tich
1           2       Tran Thi Lan   Kinh Te         34    9.2  34    Thong ke
2           7      Bui Tien Dung   Kinh Te         34    9.2  34    Thong ke


INNER JOIN chỉ giữ lại các bản ghi có khóa liên kết giữa hai bảng. Kết quả cho thấy chỉ những sinh viên có course_id hợp lệ mới được hiển thị, trong khi các bản ghi không khớp sẽ bị loại bỏ.

c. Left join

In [ ]:
df = pd.read_sql("""
SELECT *
FROM student s
LEFT JOIN course c
ON s.course_id = c.id
""", conn)
print(df)

   student_id               name     class  course_id  score    id course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12.0   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2  34.0    Thong ke
2           3       Pham Van Nam  Toan Tin        NaN    7.9   NaN        None
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2   NaN        None
4           5        Vu Quoc Anh  May Tinh       24.0    8.0   NaN        None
5           6     Dang Thuy Linh  May Tinh       24.0    5.5   NaN        None
6           7      Bui Tien Dung   Kinh Te       34.0    9.2  34.0    Thong ke
7           8        Ho Ngoc Mai  Toan Tin       20.0    8.8   NaN        None
8           9     Duong Huu Phuc   Kinh Te        NaN    7.2   NaN        None
9          10       Cao Thi Hanh  May Tinh        NaN    7.0   NaN        None


LEFT JOIN giữ toàn bộ dữ liệu từ bảng student (bảng bên trái), kể cả khi không có bản ghi tương ứng trong bảng course.
Những sinh viên không có course_id hoặc không khớp với bảng course vẫn xuất hiện, nhưng các cột của bảng course sẽ có giá trị NULL.
Các sinh viên có course_id hợp lệ sẽ được ghép đúng với tên môn học tương ứng.

d.Right join

In [ ]:
df = pd.read_sql("""
SELECT *
FROM course c
LEFT JOIN student s
ON s.course_id = c.id
""", conn)
print(df)

   id course_name  student_id               name     class  course_id  score
0  12   Giai tich         1.0  Nguyen Minh Hoang  May Tinh       12.0    6.7
1  34    Thong ke         2.0       Tran Thi Lan   Kinh Te       34.0    9.2
2  34    Thong ke         7.0      Bui Tien Dung   Kinh Te       34.0    9.2
3  26     Tin hoc         NaN               None      None        NaN    NaN


RIGHT JOIN giữ toàn bộ dữ liệu từ bảng course (bảng bên phải), kể cả khi không có sinh viên nào đăng ký môn học đó.
Những môn học chưa có sinh viên vẫn xuất hiện, nhưng các cột của bảng student sẽ có giá trị NULL.
Các môn có sinh viên đăng ký sẽ hiển thị đầy đủ thông tin sinh viên tương ứng.

e. Full outer join

In [ ]:
df = pd.read_sql("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id

UNION

SELECT * FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn)
print(df .to_string())

    student_id               name     class          course_id     score    id course_name
0            1  Nguyen Minh Hoang  May Tinh                 12       6.7  12.0   Giai tich
1            2       Tran Thi Lan   Kinh Te                 34       9.2  34.0    Thong ke
2            3       Pham Van Nam  Toan Tin               None       7.9   NaN        None
3            4     Le Thanh Huyen  Toan Tin                 20       7.2   NaN        None
4            5        Vu Quoc Anh  May Tinh                 24       8.0   NaN        None
5            6     Dang Thuy Linh  May Tinh                 24       5.5   NaN        None
6            7      Bui Tien Dung   Kinh Te                 34       9.2  34.0    Thong ke
7            8        Ho Ngoc Mai  Toan Tin                 20       8.8   NaN        None
8            9     Duong Huu Phuc   Kinh Te               None       7.2   NaN        None
9           10       Cao Thi Hanh  May Tinh               None       7.0   NaN        None

FULL OUTER JOIN giữ toàn bộ dữ liệu của cả hai bảng student và course, bao gồm cả các bản ghi không có sự liên kết.
Những sinh viên không có môn học vẫn xuất hiện (các cột course = NULL).
Những môn học chưa có sinh viên cũng vẫn xuất hiện (các cột student = NULL).
Các bản ghi có liên kết hợp lệ sẽ được ghép đầy đủ thông tin từ cả hai bảng.

2. Cập nhật course_id còn thiếu

Xóa giá trị không hợp lệ

In [ ]:
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")


Các giá trị không hợp lệ (như NULL, course_id không tồn tại, hoặc dữ liệu sai logic) đã được loại bỏ khỏi bảng.
Sau khi xóa, dữ liệu trở nên sạch hơn và nhất quán hơn, chỉ giữ lại các bản ghi có ý nghĩa.

Gán giá trị thiếu

In [ ]:
cursor.execute("""
UPDATE student
SET course_id = (
    SELECT id FROM course ORDER BY RANDOM() LIMIT 1
)
WHERE course_id IS NULL
""")

conn.commit()

Các giá trị course_id = NULL đã được cập nhật bằng cách gán ngẫu nhiên môn học
Điều này làm thay đổi phân bố dữ liệu giữa các môn



a. Tổng số SV và điểm TB theo lớp

In [ ]:
df = pd.read_sql("""
SELECT class,
       COUNT(*) AS so_sv,
       AVG(score) AS diem_tb
FROM student
GROUP BY class
""", conn)
print(df)

      class  so_sv   diem_tb
0   Kinh Te      3  8.533333
1  May Tinh      2  6.850000
2  Toan Tin      1  7.900000


b. Tổng số SV và điểm trung bình của từng môn học

In [ ]:
df = pd.read_sql("""
SELECT c.course_name,
       COUNT(*) AS so_sv,
       AVG(s.score) AS diem_tb
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)
print(df)

  course_name  so_sv   diem_tb
0   Giai tich      1  6.700000
1    Thong ke      2  9.200000
2     Tin hoc      3  7.366667


- Môn Giai tich chỉ có 1 sinh viên nen điểm trung bình thấp nhất
- Môn Thong ke có 2 sinh viên nhưng điểm trung bình cao nhất (9.2)
- Môn Tin hoc có 3 sinh viên với tổng trung bình 7.36

c. Phân loại điểm

In [ ]:
df = pd.read_sql("""
SELECT c.course_name,
       CASE
           WHEN AVG(s.score) >= 9 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6 THEN 'Tot'
           ELSE 'Kem'
       END AS xep_loai
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)
print(df)

  course_name  xep_loai
0   Giai tich       Tot
1    Thong ke  Xuat sac
2     Tin hoc       Tot


- Môn giải tích và tin học được xếp loại tốt
- Môn thống kê được xếp loại xuất sắc

3. Xếp hạng sinh viên

Sử dụng hàm RANK() để xếp hạng theo:

- Điểm số toàn bộ
- Theo lớp
- Theo môn học

a. Theo điểm số

In [ ]:
df = pd.read_sql("""
SELECT *,
       RANK() OVER (ORDER BY score DESC) AS rank_score
FROM student
""", conn)
print(df)

   student_id               name     class  course_id  score  rank_score
0           2       Tran Thi Lan   Kinh Te         34    9.2           1
1           7      Bui Tien Dung   Kinh Te         34    9.2           1
2           3       Pham Van Nam  Toan Tin         26    7.9           3
3           9     Duong Huu Phuc   Kinh Te         26    7.2           4
4          10       Cao Thi Hanh  May Tinh         26    7.0           5
5           1  Nguyen Minh Hoang  May Tinh         12    6.7           6


- Các sinh viên có cùng điểm sẽ có cùng thứ hạng (ví dụ: 9.2 → rank = 1)
- Thứ hạng có thể bị “nhảy” (1 → 3 → 4…)

b. Điểm số theo lớp học

In [ ]:
df = pd.read_sql("""
SELECT *,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_class
FROM student
""", conn)
print(df)

   student_id               name     class  course_id  score  rank_class
0           2       Tran Thi Lan   Kinh Te         34    9.2           1
1           7      Bui Tien Dung   Kinh Te         34    9.2           1
2           9     Duong Huu Phuc   Kinh Te         26    7.2           3
3          10       Cao Thi Hanh  May Tinh         26    7.0           1
4           1  Nguyen Minh Hoang  May Tinh         12    6.7           2
5           3       Pham Van Nam  Toan Tin         26    7.9           1


c. Điểm số theo mã môn học

In [ ]:
df = pd.read_sql("""
SELECT s.*, c.course_name,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_course
FROM student s
JOIN course c ON s.course_id = c.id
""", conn)
print(df.to_string())

   student_id               name     class  course_id  score course_name  rank_course
0           1  Nguyen Minh Hoang  May Tinh         12    6.7   Giai tich            1
1           3       Pham Van Nam  Toan Tin         26    7.9     Tin hoc            1
2           9     Duong Huu Phuc   Kinh Te         26    7.2     Tin hoc            2
3          10       Cao Thi Hanh  May Tinh         26    7.0     Tin hoc            3
4           2       Tran Thi Lan   Kinh Te         34    9.2    Thong ke            1
5           7      Bui Tien Dung   Kinh Te         34    9.2    Thong ke            1


d. Top 3 sinh viên đạt hạng cao nhất

In [ ]:
df = pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score DESC) AS rank
    FROM student
)
WHERE rank <= 3
""", conn)
print(df)

   student_id           name     class  course_id  score  rank
0           2   Tran Thi Lan   Kinh Te         34    9.2     1
1           7  Bui Tien Dung   Kinh Te         34    9.2     1
2           3   Pham Van Nam  Toan Tin         26    7.9     3


e. Top 3 sinh viên đạt hạng thấp nhất

In [ ]:
df = pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score ASC) AS rank
    FROM student
)
WHERE rank <= 3
""", conn)
print(df)

   student_id               name     class  course_id  score  rank
0           1  Nguyen Minh Hoang  May Tinh         12    6.7     1
1          10       Cao Thi Hanh  May Tinh         26    7.0     2
2           9     Duong Huu Phuc   Kinh Te         26    7.2     3


4. Thêm cột graduation_date

Thêm cột graduation_date kiểu DATETIME

Giá trị được tính dựa trên:
- Thời điểm hiện tại
- Thứ hạng của sinh viên

Thêm cột

In [ ]:
cursor.execute("""
ALTER TABLE student
ADD COLUMN graduation_date DATETIME
""")

In [ ]:
cursor.execute("""
UPDATE student
SET graduation_date = DATETIME('now', '+' || (
    SELECT RANK() OVER (ORDER BY score DESC)
    FROM student s2
    WHERE s2.student_id = student.student_id
) || ' days')
""")

conn.commit()

In [ ]:
df = pd.read_sql("""SELECT * FROM student""", conn)
print(df.to_string())

   student_id               name     class  course_id  score      graduation_date
0           1  Nguyen Minh Hoang  May Tinh         12    6.7  2026-04-02 16:56:50
1           2       Tran Thi Lan   Kinh Te         34    9.2  2026-04-02 16:56:50
2           3       Pham Van Nam  Toan Tin         26    7.9  2026-04-02 16:56:50
3           7      Bui Tien Dung   Kinh Te         34    9.2  2026-04-02 16:56:50
4           9     Duong Huu Phuc   Kinh Te         26    7.2  2026-04-02 16:56:50
5          10       Cao Thi Hanh  May Tinh         26    7.0  2026-04-02 16:56:50
